In [35]:
import numpy as np
from itertools import product

# Définition des fonctions de score
def score_frequence_cardiaque_temp(T_c, FC):
    if FC < 40 or T_c < 32:
        return 1
    if FC > 120 and 32 <= T_c < 35:
        return 0.9
    if FC > 140 or (FC < 50 and 32 <= T_c < 36):
        return 0.8
    if 100 <= FC <= 140 and 35 <= T_c < 36:
        return 0.7
    if 50 <= FC < 100 and 36 <= T_c <= 38.5:
        return 0.5
    if FC > 180 or T_c > 40:
        return 1
    return 0

def score_immobilite(M, t_immobile):
    if M == 0 and t_immobile > 30:
        return 1
    if M == 0 and t_immobile > 20:
        return 0.8
    if M == 0 and t_immobile > 10:
        return 0.5
    return 0

def score_temperature_environnement(T_c, T_e, FC):
    score = 0
    if T_c < 32 or T_c > 40:
        score = 1
    elif 32 <= T_c < 35 and T_e < -10:
        score = 0.9
    elif 35 <= T_c < 36 and T_e < -15:
        score = 0.8
    elif 36 <= T_c < 37 and T_e < -10:
        score = 0.7
    elif 37 <= T_c <= 38.5:
        score = 0.5
    if T_c > 39 and FC > 150:
        score = 0.9
    if T_e < -15:
        score += 0.2
    return min(1.0, score)

def score_interaction_medicale(T_c, FC, t_immobile):
    if T_c < 32 and FC < 40 and t_immobile > 20:
        return 1
    if T_c < 35 and FC > 120 and t_immobile > 15:
        return 0.8
    if T_c < 35 or t_immobile > 15:
        return 0.5
    return 0

def calculer_score_gravite(T_c, FC, M, t_immobile, T_e, w1, w2, w3, w4):
    score_fc = score_frequence_cardiaque_temp(T_c, FC)
    score_m = score_immobilite(M, t_immobile)
    score_tc = score_temperature_environnement(T_c, T_e, FC)
    score_inter = score_interaction_medicale(T_c, FC, t_immobile)
    SG = (w1 * score_fc) + (w2 * score_m) + (w3 * score_tc) + (w4 * score_inter)
    return min(1.0, SG)

# Génération de 50 scénarios de test
np.random.seed(42)
scenarios = []
for _ in range(50):
    T_c = np.random.uniform(30, 42)  # Température corporelle
    FC = np.random.randint(35, 220)  # Fréquence cardiaque
    M = np.random.choice([0, 1])  # Mouvement : 0 = immobile, 1 = mobile
    t_immobile = np.random.randint(0, 30)  # Temps d'immobilité en minutes
    T_e = np.random.uniform(-30, 15)  # Température ambiante
    SG_attendu = np.random.uniform(0.1, 1)  # Score de gravité attendu (simulé)
    scenarios.append((T_c, FC, M, t_immobile, T_e, SG_attendu))

# Grid Search sur les poids possibles
poids_possibles = np.linspace(0.1, 0.5, 5)  # Valeurs possibles pour chaque poids
meilleur_w = None
meilleure_erreur = float('inf')

def mean_squared_error(SG_predits, SG_attendus):
    return np.mean((np.array(SG_predits) - np.array(SG_attendus))**2)

for w1, w2, w3, w4 in product(poids_possibles, repeat=4):
    if abs(w1 + w2 + w3 + w4 - 1) > 0.01:
        continue  # On s'assure que la somme des poids est proche de 1
    SG_predits = [calculer_score_gravite(T_c, FC, M, t_immobile, T_e, w1, w2, w3, w4) for T_c, FC, M, t_immobile, T_e, SG in scenarios]
    SG_attendus = [SG for _, _, _, _, _, SG in scenarios]
    erreur = mean_squared_error(SG_predits, SG_attendus)
    if erreur < meilleure_erreur:
        meilleure_erreur = erreur
        meilleur_w = (w1, w2, w3, w4)

print("Meilleurs poids trouvés:", meilleur_w)
print("Erreur minimale obtenue:", meilleure_erreur)


Meilleurs poids trouvés: (np.float64(0.2), np.float64(0.2), np.float64(0.4), np.float64(0.2))
Erreur minimale obtenue: 0.08945162298080653
